# RAD Quickstart - Amazon

This notebook gives a simple, reproducible way to run the Amazon RAD pipeline from the submission folder.

It includes:
- A quick smoke run (single-seed, shorter training)
- A full ablation command
- Helpers to read output CSVs

In [ ]:
from pathlib import Path
import os

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path('/array/dahlenw/RAD/submission_2026-05-17'),
    Path('/array/dahlenw/RAD'),
]

rad_root = None
for c in candidates:
    if (c / 'scripts' / 'active' / 'rad_vs_supervised_rdl.py').exists():
        rad_root = c
        break

if rad_root is None:
    raise FileNotFoundError('Could not locate RAD project root with scripts/active/rad_vs_supervised_rdl.py')

print('RAD root:', rad_root)
print('Python   :', os.environ.get('VIRTUAL_ENV', 'system python'))

In [ ]:
import subprocess

RUN_MODE = 'smoke'  # 'smoke' or 'full'

env = os.environ.copy()
env.update({
    'SEED': '0',
    'TARGET_ANOM_RATE': '0.015',
    'POS_DOWNSAMPLE_MODE': 'subtle',
    'ATRISK_TAU_A': '0.08',
    'ATRISK_TAU_B': '6',
    'ATRISK_TAU_ALPHA': '0.05',
    'ATRISK_TAU_MULTIPLIER': '0.6',
})

if RUN_MODE == 'smoke':
    env.update({
        'EPOCHS': '15',
        'PATIENCE': '4',
        'SUP_EPOCHS': '0',
        'WARMUP_EPOCHS': '3',
    })
    cmd = ['python', str(rad_root / 'scripts' / 'active' / 'rad_vs_supervised_rdl.py')]
else:
    env.update({
        'N_SEEDS': '10',
        'START_SEED': '0',
    })
    cmd = ['python', str(rad_root / 'scripts' / 'active' / 'run_full_ablation_10seed.py')]

print('Mode   :', RUN_MODE)
print('Command:', ' '.join(cmd))

In [ ]:
# Set RUN_NOW = True to execute.
RUN_NOW = False

if RUN_NOW:
    p = subprocess.Popen(cmd, cwd=rad_root, env=env)
    rc = p.wait()
    print('Exit code:', rc)
else:
    print('Dry run only. Flip RUN_NOW=True to execute.')

In [ ]:
import pandas as pd

candidate_csvs = [
    rad_root / 'results' / 'amazon' / 'full_ablation_10seed' / 'mean_std_metrics_pretty.csv',
    rad_root / 'results' / 'amazon' / 'full_ablation_10seed' / 'mean_std_metrics.csv',
]

for p in candidate_csvs:
    if p.exists():
        print('Loaded:', p)
        display(pd.read_csv(p).head(20))
        break
else:
    print('No summary CSV found yet. Run the experiment first.')